# 02 · Data flow

**Story so far:** Review Radar can generate and normalize reviews, but they live as
in-memory lists inside task outputs. That doesn't survive scale — this chapter lands the
raw corpus in the **object store** as proper dataset artifacts and establishes how data,
credentials, and external systems flow through the platform.

**Learning goals**

1. Pass files, directories, and dataframes between tasks via the object store —
   without loading them into task inputs/outputs
2. Reference data that already lives in your buckets
3. Inject secrets as env vars or file mounts
4. (Optional) query a warehouse through a connector task
5. Compose with other teams' deployed tasks via `flyte.remote.Task.get`

In [ ]:
import flyte
from flyte.io import Dir, File

flyte.init_from_config()

env = flyte.TaskEnvironment(
    name="data_flow",
    resources=flyte.Resources(cpu="1", memory="1Gi"),
)

## 1. Files and directories

`File`/`Dir` are **references to objects in the deployment's object store** (S3, GCS, or
Azure Blob — the API is identical), not the bytes themselves. Returning a `File` uploads
once and passes a pointer; downstream tasks stream or download on demand. This is the
single most important I/O rule in Flyte:

> Large data goes through `File`/`Dir`/`DataFrame`. Task inputs/outputs (strings, lists,
> dicts) are for *coordination*, not payloads — oversized inline values slow every hop
> and can OOM the orchestrator.

Task pods reach the store through the deployment's identity binding (IRSA on AWS,
Workload Identity on GCP, workload identities on Azure) — no keys in code, ever.

Let's land the raw Review Radar corpus as a JSONL file:

In [ ]:
import json
import random

PRODUCTS = ["espresso machine", "trail shoes", "noise-canceling headphones", "air fryer"]
PHRASES = {
    5: "absolutely love this {p}, exceeded expectations",
    4: "solid {p}, works as advertised",
    3: "the {p} is okay, nothing special",
    2: "disappointed with the {p}, quality issues",
    1: "terrible {p}, arrived broken and support ignored me",
}


@env.task
async def land_raw_reviews(n: int, seed: int = 7) -> File:
    """Write the raw corpus to the object store; return a reference, not the data."""
    rng = random.Random(seed)
    with open("raw_reviews.jsonl", "w") as f:
        for i in range(n):
            stars = rng.randint(1, 5)
            p = rng.choice(PRODUCTS)
            f.write(json.dumps({"id": i, "product": p, "stars": stars,
                                "text": PHRASES[stars].format(p=p)}) + "\n")
    return await File.from_local("raw_reviews.jsonl")


@env.task
async def dataset_summary(raw: File) -> dict:
    counts: dict = {}
    async with raw.open("r") as fh:            # streams from the object store
        content = await fh.read()
    for line in content.splitlines():
        stars = json.loads(line)["stars"]
        counts[stars] = counts.get(stars, 0) + 1
    return counts


@env.task
async def land_and_summarize(n: int = 500) -> dict:
    raw = await land_raw_reviews(n=n)
    print(f"raw corpus at: {raw.path}")
    return await dataset_summary(raw=raw)


run = await flyte.run.aio(land_and_summarize, n=500)
print(run.url)
await run.wait.aio()
run.outputs()

In [ ]:
# Data that ALREADY lives in the customer's buckets — reference it, nothing is copied.
# Works with any scheme the deployment's store speaks: s3://, gs://, abfs://
from workshop_config import WS                      # client-side values only

if WS.object_store:
    existing = File.from_existing_remote(f"{WS.object_store}/path/to/data.csv")

# Create a remote reference to write to, then hand it downstream:
# out = File.new_remote()

# Directories work the same way:
# d = Dir.from_existing_remote(f"{WS.object_store}/training-data/")
# async for f in d.walk(): ...

## 2. DataFrames

`flyte.io.DataFrame` is a lazy, format-aware reference (Parquet in the object store by
default) that converts at the edges — a pandas producer can feed a polars consumer.
Review Radar's tabular view of the corpus:

In [ ]:
from flyte.io import DataFrame

df_image = (
    flyte.Image.from_debian_base(name="workshop-df", python_version=(3, 12))
    .with_pip_packages("pandas==2.2.3", "pyarrow")
)

df_env = flyte.TaskEnvironment(
    name="dataframes",
    image=df_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
)


@df_env.task
async def reviews_to_frame(raw: File) -> DataFrame:
    import json
    import pandas as pd
    rows = []
    async with raw.open("r") as fh:
        for line in (await fh.read()).splitlines():
            rows.append(json.loads(line))
    return DataFrame.from_df(pd.DataFrame(rows))


@df_env.task
async def stars_by_product(df: DataFrame) -> dict:
    import pandas as pd
    pdf = await df.open(pd.DataFrame).all()
    return pdf.groupby("product")["stars"].mean().round(2).to_dict()

## 3. Secrets

Secrets are stored by the platform and **injected at task startup** as env vars or file
mounts. Code never sees the secret store. Review Radar will need one the moment its
ingest hits a real API:

```bash
flyte create secret REVIEWS_API_KEY <value>
flyte create secret --from-file tls.crt CA_CERT
flyte get secret
```

In [ ]:
import os

secret_env = flyte.TaskEnvironment(
    name="with_secrets",
    resources=flyte.Resources(cpu="1", memory="512Mi"),
    secrets=[
        flyte.Secret(key="REVIEWS_API_KEY", as_env_var="API_KEY"),
        # file mount, for certs/keys that shouldn't be in env vars:
        # flyte.Secret(key="CA_CERT", mount="/etc/workshop/secrets"),
    ],
)


@secret_env.task
async def fetch_from_source() -> str:
    key = os.environ["API_KEY"]              # injected by the platform
    return f"would call the reviews API with a key of length {len(key)}"

Registry credentials for private images use the same machinery — a secret name passed as
`registry_secret` to `flyte.Image` ([01](./01-authoring-fundamentals.ipynb) §3).

> 💬 **Discuss:** which data sources will the real ingest touch first (APIs, buckets,
> warehouse tables), and which identities own that access today — platform-level identity
> bindings or per-source credentials that belong in Flyte secrets?

## 4. Optional: warehouse connectors

If the raw reviews live in a warehouse, **connector tasks** query it without running in
your container — the query executes on the connector service in the data plane, which
submits and polls without holding a worker. BigQuery is shown here; Snowflake and
Databricks connectors follow the same declare-a-task shape (swap this section to match
the customer's warehouse).

In [ ]:
# Requires the BigQuery connector enabled in the deployment (appendix A) and
# WAREHOUSE_PROJECT set in .env. Skip this cell otherwise.
from flyte.io import DataFrame
from flyteplugins.bigquery import BigQueryConfig, BigQueryTask

bq_config = BigQueryConfig(
    ProjectID=WS.warehouse_project or "<project>",
    Location="US",
)

reviews_from_warehouse = BigQueryTask(
    name="reviews_from_warehouse",
    query_template="""
        SELECT id, product, stars, text
        FROM dataset.reviews
        WHERE stars <= @max_stars
        LIMIT 1000
    """,
    plugin_config=bq_config,
    inputs={"max_stars": int},                    # typed query parameters
    output_dataframe_type=DataFrame,
)

bq_env = flyte.TaskEnvironment.from_task("bq_env", reviews_from_warehouse)
# run = await flyte.run.aio(reviews_from_warehouse, max_stars=2)

## 5. Cross-team composition: remote tasks

Teams publish tasks by deploying them; consumers reference by **name**, without importing
code or dependencies. First deploy the example shared task (once):

In [ ]:
!python scripts/remote_task_deploy.py

In [ ]:
import flyte.remote

tokenize = flyte.remote.Task.get("shared_utils.tokenize", auto_version="latest")


@env.task
async def use_shared(text: str) -> int:
    tokens = await tokenize(text=text)
    return len(tokens)


run = await flyte.run.aio(use_shared, text="composed from another team's task")
print(run.url)
await run.wait.aio()
run.outputs()

**Story checkpoint:** the raw corpus now lives in the object store as a durable, versioned
artifact, and we know how credentials and external sources plug in. Next: process it — all
of it, in parallel.

## Further reading

- Next: [03-processing-at-scale](./03-processing-at-scale.ipynb)